# Préparation df_irve

In [ ]:
import pandas as pd

In [ ]:
df_irve = pd.read_csv(
 'https://www.data.gouv.fr/api/1/datasets/r/eb76d20a-8501-400e-b336-d85724de5435'
)

print(f"Shape : {df_irve.shape}")
df_irve.sample(5)

Ce premier jeu de données est la base nationale des IRVE (Infrastructures de Recharge pour Véhicules Électriques).
Cette table comporte 212234 lignes pour 52 variables.

In [ ]:
pip install shapely

In [ ]:
pip install geopandas

In [ ]:
pip install cartiflette

In [ ]:
from src.preparation_data import nettoyer_code_insee
df_irve["code_insee_commune"] = df_irve["code_insee_commune"].apply(nettoyer_code_insee)

In [ ]:
from src.preparation_data import creer_gdf_irve, charger_communes, joindre_communes, ajouter_codes_geo

In [ ]:
gdf_irve = creer_gdf_irve(df_irve, "consolidated_longitude", "consolidated_latitude")
gdf_communes = charger_communes()
gdf_result = joindre_communes(gdf_irve, gdf_communes)
df_irve = ajouter_codes_geo(df_irve, gdf_result)

# Choix des variables

In [ ]:
list(df_irve.columns)

Après un leger tri des variables disponibles, on choisit de s'interresser aux suivantes :

['nom_amenageur',
 'nom_operateur',
 'nom_enseigne',
 'id_station_itinerance',
 'id_station_local',
 'id_pdc_itinerance',
 'id_pdc_local',
 'implantation_station',
 'nbre_pdc',
 'puissance_nominale',
 'prise_type_ef',
 'prise_type_2',
 'prise_type_combo_ccs',
 'prise_type_chademo',
 'prise_type_autre',
 'cable_t2_attache',
 'gratuit',
 'paiement_acte',
 'paiement_cb',
 'paiement_autre',
 'tarification',
 'condition_acces',
 'reservation',
 'horaires',
 'created_at']

In [ ]:
var_interet = ['nom_amenageur',
               'nom_operateur',
               'nom_enseigne',
               'id_station_itinerance',
               'id_station_local',
               'id_pdc_itinerance',
               'id_pdc_local',
               'implantation_station',
               'nbre_pdc',
               'puissance_nominale',
               'prise_type_ef',
               'prise_type_2',
               'prise_type_combo_ccs',
               'prise_type_chademo',
               'prise_type_autre',
               'cable_t2_attache',
               'gratuit',
               'paiement_acte',
               'paiement_cb',
               'paiement_autre',
               'tarification',
               'condition_acces',
               'reservation',
               'horaires',
               'created_at']

ne pas oublier d'ajouter `code_geo_total`

1. Variables d'identification et d'acteurs
Ces variables permettent de mesurer la diversité et le type d'acteurs présents.

Représentent les entités responsables de l'installation et de l'exploitation.
Usage final : Calculez le nombre d'opérateurs différents par commune. Plus il y a d'acteurs, plus le réseau est compétitif et attractif.
['nom_amenageur',
 'nom_operateur',
 'nom_enseigne']


2. Variables de localisation et d'infrastructure

Identifiants uniques de la station (un regroupement de plusieurs bornes).
Usage final : Comptez le nombre de stations par commune (plutôt que le nombre de points de charge) pour mesurer le maillage territorial.
 'id_station_itinerance',
 'id_station_local',

Identifiants du point de charge précis.
'id_pdc_itinerance',
 'id_pdc_local',

Type de lieu (voirie, parking public, parking privé, etc.).
Usage final : Calculez la part des bornes en voirie (accessible 24/24) vs en parking.
 'implantation_station',

Nombre de points de charge sur la borne.
Usage final : À sommer par commune pour obtenir la capacité totale d'accueil.
 'nbre_pdc',


3. Puissance et Types de prises (Performance technique)
C'est le cœur de l'attractivité pour l'utilisateur.

Puissance délivrée (en kW).
Usage final : Calculez la puissance moyenne ou créez une variable "Nombre de bornes ultra-rapides" (> 50 kW).
 'puissance_nominale',

Indiquent les standards de recharge disponibles.
Usage final : Calculez le taux de compatibilité Combo CCS (standard européen pour la charge rapide). Une commune n'ayant que du Type EF (prises domestiques) est moins attractive.
 'prise_type_ef',
 'prise_type_2',
 'prise_type_combo_ccs',
 'prise_type_chademo',
 'prise_type_autre',
 'cable_t2_attache',


4. Services, Tarification et Accessibilité
Ces variables influencent directement le coût d'usage.

Indique si la recharge est gratuite.
Usage final : Taux de gratuité par commune. Un fort taux peut booster l'achat de VE locaux.
 'gratuit',

Modalités de paiement.
Usage final : Taux d'acceptation CB. Le paiement direct est un facteur clé de simplicité.
 'paiement_acte',
 'paiement_cb',
 'paiement_autre',
 'tarification',

Accessibilité de la borne.
Usage final : Créez une variable binaire sur la disponibilité 24h/24.
 'condition_acces',
 'reservation',
 'horaires',

Date de création de la fiche.
Usage final : Calculez l'ancienneté moyenne du réseau par commune.
 'created_at']

In [ ]:
df_filtre = df_irve[var_interet]

type de chaque variable

In [ ]:
df_filtre.dtypes

Les types de données fournis par la source ne sont pas respectés dans les données brutes. Plusieurs variables booléennes et temporelles sont encodées en texte (object), nécessitant une étape de conversion avant analyse.

1. convertir les booleens

In [ ]:
cols_bool = [
    'prise_type_ef',
    'prise_type_2',
    'prise_type_combo_ccs',
    'prise_type_chademo',
    'prise_type_autre',
    'cable_t2_attache',
    'gratuit',
    'paiement_acte',
    'paiement_cb',
    'paiement_autre',
    'reservation'
]

In [ ]:
for col in cols_bool:
    print(f"\n----- {col} -----")
    print(df_filtre[col].value_counts(dropna=False))

In [ ]:
# Récupérer toutes les valeurs uniques
valeurs_uniques = set()

for col in cols_bool:
    valeurs_uniques.update(df_filtre[col].unique())

# Afficher le résultat
print(valeurs_uniques)

In [ ]:
import numpy as np

mapping = {
    'true': True,
    'false': False,
    '1': True,
    '0': False
}

for col in cols_bool:
    df_filtre[col] = (
        df_filtre[col]
        .astype(str)
        .str.strip()
        .str.lower()
        .map(mapping)
        .astype("boolean")
    )

In [ ]:
for col in cols_bool:
    print(f"{col} :", df_filtre[col].unique())

Convertir la date

In [ ]:
df_filtre['created_at'] = pd.to_datetime(df_filtre['created_at'])

Convertir string

In [ ]:
cols_str = [
    'nom_amenageur',
    'nom_operateur',
    'nom_enseigne',
    'id_station_local',
    'id_pdc_itinerance',
    'id_station_itinerance',
    'id_pdc_local',
    'implantation_station',
    'tarification',
    'condition_acces',
    'horaires'
]

for col in cols_str:
    df_filtre[col] = (
        df_filtre[col]
        .astype("string")
    )

Verification des types

In [ ]:
df_filtre.dtypes

In [ ]:
df_filtre.describe(include='string[python]')

'implantation_station' et 'condition_acces' ont peu de valeurs uniques. On les convertit donc en 'category'

In [ ]:
df_filtre['implantation_station'] = df_filtre['implantation_station'].astype('category')
df_filtre['condition_acces'] = df_filtre['condition_acces'].astype('category')

finalement :

In [ ]:
df_filtre.describe(include='all')

In [ ]:
df_filtre.isna().sum()

In [ ]:
(df_filtre.isna().sum() / len(df_filtre) * 100).sort_values(ascending=False).round(2).head(5)

Les colonnes avec BEAUCOUP de valeurs manquantes :

| Colonne            | Nb manquants | Interprétation     |
| ------------------ | ------------ | ------------------ |
| `cable_t2_attache` | 102 007      | très problématique |
| `gratuit`          | 34 365       | moyen              |
| `paiement_autre`   | 39 806       | moyen              |
| `paiement_cb`      | 15 944       | acceptable         |
| `nom_operateur`    | 3 992        | faible             |


cable_t2_attache : Présence d’un câble Type 2 attaché directement à la borne (oui/non).
Donc :
True → câble déjà présent
False → il faut son propre câble

Presque 50% de NA, on fait le choix de ne pas utiliser cette variable.

paiement_autre : Autres moyens de paiement (badge, appli, etc.)
Utile mais moins précise.
Pas mal de NA.
Variable floue ("autre"), optionnelle pour le ML, voir analyse descriptive.

Voyons en quoi nos variables pourraient nous servir :


4. Services, Tarification et Accessibilité
Ces variables influencent directement le coût d'usage.

Modalités de paiement.
Usage final : Taux d'acceptation CB. Le paiement direct est un facteur clé de simplicité.
 'paiement_acte',
 'tarification',

Accessibilité de la borne.
Usage final : Créez une variable binaire sur la disponibilité 24h/24.
 'condition_acces',
 'reservation',
 'horaires',

Date de création de la fiche.
Usage final : Calculez l'ancienneté moyenne du réseau par commune.
 'created_at'

In [ ]:
# nbre_pdc
sum_nbre_pdc = df.groupby("code_geo")["nbre_pdc"].sum()
mean_nbre_pdc = df.groupby("code_geo")["nbre_pdc"].mean()

In [ ]:
# puissance_nominale
df.groupby("code_geo")["puissance_nominale"].agg(["mean", "max"])
# faire des analyse uni pour bien comprendre les puissances des bornes et  créez une variable "Nombre de bornes ultra-rapides" (> 50 kW)

In [ ]:
# booléenne : proportions -> à faire pour toutes -> obtient le taux par code géo
# ex :
df.groupby("code_geo")["prise_type_2"].mean()

In [ ]:
# var catégorielle :
# one-hot encoding + moyenne (éventuellement que pour certaines modalités ou regrouper des modalités pour créer une nouvelle)
# ex ?
df.groupby("code_geo")["implantation_station"].value_counts(normalize=True)

In [ ]:
# nom_operateur
# top opérateur
df.groupby("code_geo")["nom_operateur"].agg(lambda x: x.mode()[0])
# nb d'opérateurs : mesure de concurrence
df.groupby("code_geo")["nom_operateur"].nunique()

In [ ]:
# created_at
df["annee"] = df["created_at"].dt.year
df.groupby("code_geo")["annee"].mean()
# ou
# nombre de bornes récentes
# évolution

variables à exclure :
❌ Identifiants
id_station_*
id_pdc_*
👉 aucune valeur prédictive

❌ nom_amenageur, nom_enseigne
👉 trop de modalités → bruit

❌ horaires, tarification
👉 texte libre → difficile à exploiter
!!! faire des analyses univariées quand même pour voir !!!

❌ prise_type_autre
❌ cable_t2_attache


Nouvelle sélection :

In [ ]:
var_interet = ['nom_operateur',
               'implantation_station',
               'nbre_pdc',
               'puissance_nominale',
               'prise_type_ef',
               'prise_type_2',
               'prise_type_combo_ccs',
               'prise_type_chademo',
               'gratuit',
               'paiement_acte',
               'paiement_cb',
               'paiement_autre',
               'tarification',
               'condition_acces',
               'reservation',
               'horaires',
               'created_at']

Au final, pour chaque code géo :

Pour les vairables quanti :
total_pdc
puissance_moyenne
puissance_max

Pour le marché / les acteurs :
top_operateur
nb_operateurs

Infrastructure :
% type_2
% combo_ccs
% chademo
% type_ef

Accessibilité :
% gratuit
% paiement_cb
% paiement_autre

Environnement :
% voirie / parking / ... (avec 'implantation_station')

